In [18]:
import heapq

class State:
    def __init__(self, board, g, h, parent=None, move_dir=None):
        self.board = tuple(board)  # 1D tuple of length 9
        self.g = g
        self.h = h
        self.f = g + h
        self.parent = parent
        self.move_dir = move_dir  # The direction moved to reach this state

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

def calculate_heuristic(current_board, goal_board):
    # Heuristic: Count of misplaced tiles (excluding the empty tile '0')
    misplaced = 0
    for i in range(9):
        if current_board[i] != 0 and current_board[i] != goal_board[i]:
            misplaced += 1
    return misplaced

def get_neighbors(state, goal_board):
    neighbors = []
    board = list(state.board)
    zero_idx = board.index(0)
    row, col = zero_idx // 3, zero_idx % 3

    # Moves mapping: offset, direction label
    moves = []
    if row > 0: moves.append((-3, "Up"))
    if row < 2: moves.append((3, "Down"))
    if col > 0: moves.append((-1, "Left"))
    if col < 2: moves.append((1, "Right"))

    for offset, direction in moves:
        new_board = list(board)
        target_idx = zero_idx + offset
        # Swap empty tile with the target tile
        new_board[zero_idx], new_board[target_idx] = new_board[target_idx], new_board[zero_idx]

        g_new = state.g + 1
        h_new = calculate_heuristic(new_board, goal_board)
        neighbors.append(State(new_board, g_new, h_new, parent=state, move_dir=direction))

    return neighbors

def a_star_search(initial_board, goal_board):
    initial_h = calculate_heuristic(initial_board, goal_board)
    start_state = State(initial_board, 0, initial_h)

    open_list = []
    heapq.heappush(open_list, start_state)
    closed_set = set()

    nodes_explored = 0

    while open_list:
        current_state = heapq.heappop(open_list)
        nodes_explored += 1

        if current_state.board == tuple(goal_board):
            return current_state, nodes_explored

        closed_set.add(current_state.board)

        for neighbor in get_neighbors(current_state, goal_board):
            if neighbor.board in closed_set:
                continue

            heapq.heappush(open_list, neighbor)

    print("No path to goal found.")
    return None, nodes_explored

def print_path(solution_state):
    path = []
    curr = solution_state
    while curr:
        path.append(curr)
        curr = curr.parent
    path.reverse()

    print("\n--- Solution Path Summary ---")
    for i, node in enumerate(path):
        direction_str = f" | {node.move_dir}" if node.move_dir else " | Start"
        print(f"Step {i}: State = {list(node.board)} | g = {node.g}, h = {node.h}, f(n) = {node.f}{direction_str}")

# Get state inputs from user for 8-puzzle
print("Enter 9 numbers (0-8) separated by spaces where 0 represents the blank tile.")
try:
    init_input = input("Enter Initial State (e.g., 1 2 3 0 4 6 7 5 8): ").strip().split()
    initial_board = [int(x) for x in init_input]

    goal_input = input("Enter Goal State (e.g., 1 2 3 4 5 6 7 8 0): ").strip().split()
    goal_board = [int(x) for x in goal_input]

    if len(initial_board) != 9 or len(goal_board) != 9 or set(initial_board) != set(range(9)) or set(goal_board) != set(range(9)):
        raise ValueError("Each board must contain exactly unique numbers from 0 to 8.")

    solution, steps = a_star_search(initial_board, goal_board)
    if solution:
        print_path(solution)
except Exception as e:
    print(f"\nError or empty input: {e}.")
    print("Running default example:")
    # Default standard reachable 8-puzzle state configuration
    initial_board = [1, 2, 3, 0, 4, 6, 7, 5, 8]
    goal_board = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    print(f"Initial: {initial_board}")
    print(f"Goal: {goal_board}")

    solution, steps = a_star_search(initial_board, goal_board)
    if solution:
        print_path(solution)


Enter 9 numbers (0-8) separated by spaces where 0 represents the blank tile.
Enter Initial State (e.g., 1 2 3 0 4 6 7 5 8): 2 8 3 1 6 4 7 0 5
Enter Goal State (e.g., 1 2 3 4 5 6 7 8 0): 1 2 3 8 0 4 7 6 5

--- Solution Path Summary ---
Step 0: State = [2, 8, 3, 1, 6, 4, 7, 0, 5] | g = 0, h = 4, f(n) = 4 | Start
Step 1: State = [2, 8, 3, 1, 0, 4, 7, 6, 5] | g = 1, h = 3, f(n) = 4 | Up
Step 2: State = [2, 0, 3, 1, 8, 4, 7, 6, 5] | g = 2, h = 3, f(n) = 5 | Up
Step 3: State = [0, 2, 3, 1, 8, 4, 7, 6, 5] | g = 3, h = 2, f(n) = 5 | Left
Step 4: State = [1, 2, 3, 0, 8, 4, 7, 6, 5] | g = 4, h = 1, f(n) = 5 | Down
Step 5: State = [1, 2, 3, 8, 0, 4, 7, 6, 5] | g = 5, h = 0, f(n) = 5 | Right
